In [41]:
import json,time
import ollama
import re

from pageindex import PageIndexClient

In [42]:
#
PDF_PATH ="/home/girl/Documents/coding/jupiter/vectorless_rag/HTML_Structure_Task_Report_akshat.pdf"
OLLAMA_MODEL= "ministral-3:3b-cloud"

PAGEINDEX_API_KEY = "ccbe308572444f36bb7cf6324697739d"
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)


In [43]:
def llm(prompt):
    return ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])["message"]["content"]

In [44]:
def build_tree(pdf):
    doc_id=pi.submit_document(pdf)["doc_id"]
    while True:
        s=pi.get_document(doc_id).get("status")
        print("status:",s)
        if s == "completed": break
        if s == "failed":raise RuntimeError("Processing failed")
        time.sleep(5)
    return doc_id, pi.get_tree(doc_id,node_summary=True).get("result",[])
    
        
        

In [45]:
def slim(nodes):
    return [{"node_id":n["node_id"],"title":n["title"],"page":n.get("page_index"),
             "summary":n.get("text","")[:120],**({"children":slim(n["nodes"])}
             if n.get("nodes") else {})}
             for n in nodes]
             
                                                                    

In [46]:
def find_nodes(tree,ids):
    found=[]
    for n in tree:
        if n["node_id"] in ids: found.append(n)
        if n.get("nodes"): found.extend(find_nodes(n["nodes"],ids))
    return found

In [47]:
def ask(query, tree):
    raw = llm(f"""Given this document tree, return the node IDs most relevant to the query.
Query: {query}
Tree: {json.dumps(slim(tree))}
Reply ONLY as JSON: {{"thinking":"...","node_list":["id1","id2"]}}""")

    # clean control characters and code fences
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    clean = re.sub(r'[\x00-\x1f\x7f]', ' ', clean)  # remove control chars
    clean = clean[clean.find("{"):clean.rfind("}")+1]  # extract JSON block

    try:
        ids = json.loads(clean).get("node_list", [])
    except:
        ids = []  # fallback: no nodes found, answer from empty context

    nodes   = find_nodes(tree, ids)
    context = "\n---\n".join(f"[{n['title']} | p.{n.get('page_index','?')}]\n{n.get('text','')}" for n in nodes)
    answer  = llm(f"Answer using ONLY this context, cite section+page.\nQ: {query}\n\n{context}")
    print(f"\nQ: {query}\nA: {answer}\n")
    return answer

In [ ]:
if __name__ == "__main__":
    doc_id, tree = build_tree(PDF_PATH)   # builds tree from PageIndex
    while q := input("Question (quit to exit): ").strip():
        if q == "quit": break
        ask(q, tree)
        

status: processing
status: completed


Question (quit to exit):  why creates ambiguity in this process ?



Q: why creates ambiguity in this process ?
A: Based on the provided context, ambiguity in heading hierarchies arises primarily due to **design-driven deviations from semantic structure**, which disrupts both **SEO and UX** interpretations. Key reasons include:

1. **Skipping Hierarchy Levels** (p.4)
   - **Example 1 (E-Commerce):** Skipping H2/H3 for spec labels (e.g., jumping from H1 to H4/H5) confuses search engines about content priority, reducing structured data eligibility for featured snippets.
   - **Example 2 (News/Blog):** Sidebar H2s (e.g., "Related Articles") appear before article H2s, creating semantic noise that dilutes the article’s topic modeling.
   - **Example 3 (Corporate):** H4s replace H3s in feature cards, breaking the expected H1 > H2 > H3 hierarchy, which harms keyword targeting.

2. **Semantic Misalignment** (p.4)
   - **Legacy Themes:** Using H2 for sidebar elements (e.g., "Categories") before article H2s violates logical flow, confusing crawlers and screen re

Question (quit to exit):  exit



Q: exit
A: The **exit** (or lack thereof) from a proper heading hierarchy—such as skipping levels (e.g., H1 → H4)—is primarily a **design-driven issue** with measurable consequences across **SEO, accessibility, and UX**, as detailed in the report.

### **Key Implications of Skipped Headings (Citing Context)**
1. **SEO Impact (p.2, p.5)**
   - Search engines (e.g., Googlebot) struggle to interpret the **topic structure** of content, weakening on-page signals for classification and ranking.
   - Pages with broken hierarchies (e.g., H1 → H4) are **less likely to qualify for featured snippets** or Google’s **Passage Indexing**, as structured data lacks logical subordination (e.g., H2/H3).
   - Indirect SEO harm includes **reduced accessibility** (screen readers misinterpret outlines) and **lower engagement signals** (e.g., higher bounce rates due to confusing navigation).

2. **UX & Accessibility (p.4, p.5)**
   - **Screen reader users** encounter **disorienting navigation**, as the logic